# SRQ-FLY Priority 5 — whole-process GPU memory

Train-only CIFAR-100 audit. Exact FLY and locked P2B SRQ each run from frozen ViT loading through ten analytic updates in an isolated process. No test split is opened.

In [ ]:
# Fresh repository and dependencies.
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
REPO_GIT_URL='https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH='experiment/soho-selfcontained'
WORK_DIR='/content/SOHO-CL'
# The kernel may still be inside WORK_DIR from an earlier run. Move out
# before deleting it; otherwise Git cannot resolve the current directory.
os.chdir('/content')
repo=Path(WORK_DIR)
if repo.exists(): shutil.rmtree(repo)
clone=subprocess.run(['git','clone','--depth','1','--branch',REPO_BRANCH,'--single-branch',REPO_GIT_URL,WORK_DIR],text=True,capture_output=True)
if clone.returncode!=0:
    print('CLONE STDOUT:\n',clone.stdout); print('CLONE STDERR:\n',clone.stderr)
    raise RuntimeError('Repository clone failed; inspect CLONE STDERR above.')
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','kagglehub','huggingface_hub','nvidia-ml-py'],check=True)
print('REPO COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())

In [ ]:
# Locked identities. This cell must pass before any expensive work.
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
CONFIG='configs/srq_fly_priority5_cifar100_whole_process_memory.json'
RUNNER='tools/srq_fly_priority5_memory.py'
EXPECTED={
 'configs/srq_fly_priority5_cifar100_whole_process_memory.json':'ba02e0e742fdaf17e364d6ef182d8e32f8220ac5140253ec2155572054db75da',
 'tools/srq_fly_priority5_memory.py':'eb8bb1429de804894872936d3e5c8e6beba55706ad224f7ce84048ec54fd8019',
 'methods/srq_fly_optimized/learner.py':'40edac2e2cc88faac549f5c87217f3143d815bf53ecad8a37dfdb22c112691ae',
 'methods/srq_fly_optimized/storage.py':'9d288a3661985da657371e8581f406825d4a8d5e6e0c63381aacda8484490986',
 'tools/srq_fly_system_benchmark.py':'85e2d7f8a27f8081148a88bf88dcd374af20aa39f2a77691c9ca744a2ff0d96c',
 'models/backbone.py':'941e449dc6e66ca4018fb0d3ab3218d97ec97f498b557ed220c8332e75850a46',
 'models/flyhash.py':'24ba321a71f735031b0da430ab4d3519e54e6c3149fc6c913c63b4172f6712cb',
 'utils/data_utils.py':'3cf85993e231b068ad5ae2f96be608b2e50e9c52f98fb2387fd3badfb44b6764',
 'utils/train_utils.py':'e24983bd3042ad82ec069916ba2853cf1c818cb2911ce193710c8ccd70e86bda'}
for path,expected in EXPECTED.items(): assert sha(path)==expected,(path,sha(path),expected)
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip(),'Repository must start clean.'
print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],text=True).strip())
print('LOCKED SOURCE CHECK: PASS')

In [ ]:
# Synthetic/unit correctness only; no dataset is opened.
subprocess.run([sys.executable,'-m','pytest','-q','-p','no:cacheprovider','tests/test_srq_fly_priority5_memory.py','tests/test_srq_fly_priority4_task_frequency.py','tests/test_srq_fly_priority2d_equivalence.py'],check=True)
print('PRIORITY-5 CORRECTNESS GATE: PASS')

In [ ]:
# Download the public CIFAR training source and the locked frozen checkpoint.
import kagglehub
from huggingface_hub import hf_hub_download
CIFAR_ROOT=kagglehub.dataset_download('zaphat206/cifar-100')
CHECKPOINT_PATH=hf_hub_download('timm/vit_base_patch16_224.augreg2_in21k_ft_in1k','model.safetensors')
assert Path(CIFAR_ROOT).exists()
assert Path(CHECKPOINT_PATH).stat().st_size==346284714
assert sha(CHECKPOINT_PATH)=='32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
print('CIFAR ROOT:',CIFAR_ROOT)
print('CHECKPOINT PASS:',CHECKPOINT_PATH)
# CPU preflight executes the exact loader/transform path before the long GPU run.
from tools.srq_fly_priority5_memory import _NVMLSampler, _build_train_loader, _read_config
preflight_root=Path('/content/srq_priority5_loader_preflight')
loader=_build_train_loader(_read_config(Path(CONFIG)),Path(CIFAR_ROOT),preflight_root)
images,labels=next(iter(loader))
assert tuple(images.shape[1:])==(3,224,224) and images.dtype.is_floating_point
assert labels.ndim==1 and len(images)==len(labels)
del images,labels,loader
shutil.rmtree(preflight_root)
print('TRAIN LOADER/TRANSFORM PREFLIGHT: PASS')
sampler=_NVMLSampler(0)
device_bytes,process_bytes=sampler.sample(os.getpid())
print('NVML PREFLIGHT: PASS |',sampler.device_name,'| device MiB=',device_bytes/2**20,'| parent CUDA context=',process_bytes)
sampler.close()
assert process_bytes is None,'Parent notebook already owns a CUDA context; restart runtime for an uncontaminated isolated-process measurement.'

In [ ]:
# Long cell: two fresh end-to-end worker processes with live STAGE/TASK lines.
OUTPUT_DIR='/content/srq_priority5_memory'
SCRATCH_DIR='/content/srq_priority5_scratch'
for path in (OUTPUT_DIR,SCRATCH_DIR):
    if Path(path).exists(): shutil.rmtree(path)
command=[sys.executable,'-u',RUNNER,'run','--config',CONFIG,'--root',CIFAR_ROOT,'--backbone-checkpoint',CHECKPOINT_PATH,'--output-dir',OUTPUT_DIR,'--scratch-dir',SCRATCH_DIR,'--device','cuda','--require-clean-git']
print('PRIORITY-5 START: Exact FLY then SRQ; feature extraction is repeated intentionally.',flush=True)
LOG_PATH='/content/srq_priority5_runner.log'
with open(LOG_PATH,'w',encoding='utf-8') as log:
    process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process.stdout:
        print(line,end='',flush=True); log.write(line); log.flush()
    return_code=process.wait()
print('PRIORITY-5 RETURN CODE:',return_code,'| log:',LOG_PATH)
assert return_code==0,'Priority 5 failed; return the complete live output without relaxing gates.'
RESULT=Path(OUTPUT_DIR)/'priority5_memory_results.json'
payload=json.loads(RESULT.read_text())
print('PRIORITY-5 DECISION:',payload['status'])

In [ ]:
# Compact numerical audit.
import pandas as pd
rows=[]
monitor={row['method']:row for row in payload['nvml']}
def stage_process_mib(row,stage):
    value=row.get('stage_peaks',{}).get(stage,{}).get('process_bytes')
    return float('nan') if not value else value/2**20
for row in payload['methods']:
    method=row['method']; nv=monitor[method]
    rows.append({'method':method,'persistent_MiB':row['persistent_state_bytes']/2**20,'torch_analytic_peak_MiB':row['torch_cuda_stages']['analytic_update']['peak_allocated_bytes']/2**20,'nvml_analytic_peak_MiB':stage_process_mib(nv,'analytic_update'),'nvml_whole_process_peak_MiB':nv['peak_worker_process_bytes']/2**20,'feature_extraction_peak_MiB':stage_process_mib(nv,'feature_extraction'),'update_seconds':sum(row['task_update_seconds']),'solver_residual':row['solver_relative_residual']})
table=pd.DataFrame(rows)
display(table)
print(json.dumps(payload['comparisons'],indent=2))
print(json.dumps(payload['gates'],indent=2))

In [ ]:
# Paper-facing visualization: keep state, allocator, NVML stage, and whole process distinct.
import matplotlib.pyplot as plt
labels=['Exact FLY','SRQ-FLY P2B']; colors=['#4C78A8','#F58518']
columns=[('persistent_MiB','Persistent state'),('torch_analytic_peak_MiB','PyTorch analytic peak'),('nvml_analytic_peak_MiB','NVML analytic peak'),('nvml_whole_process_peak_MiB','NVML whole-process peak')]
fig,axes=plt.subplots(1,4,figsize=(15,3.5))
for axis,(column,title) in zip(axes,columns):
    values=table[column].tolist(); bars=axis.bar(labels,values,color=colors)
    axis.set_title(title); axis.set_ylabel('MiB'); axis.tick_params(axis='x',rotation=20)
    axis.bar_label(bars,fmt='%.0f',padding=2,fontsize=8)
fig.suptitle('SRQ-FLY memory audit: distinct measurement scopes')
fig.tight_layout(); FIGURE=Path('/content/srq_fly_priority5_memory.png'); fig.savefig(FIGURE,dpi=180,bbox_inches='tight'); plt.show()

In [ ]:
# Export evidence only; dataset, checkpoint, scratch features, and symlink view are excluded.
BUNDLE=Path('/content/srq_fly_priority5_whole_process_memory')
if BUNDLE.exists(): shutil.rmtree(BUNDLE)
BUNDLE.mkdir()
shutil.copytree(OUTPUT_DIR,BUNDLE/'results')
shutil.copy2(CONFIG,BUNDLE/'locked_config.json')
shutil.copy2('docs/research/SRQ_FLY_PRIORITY5_MEMORY_PROTOCOL.md',BUNDLE/'protocol.md')
shutil.copy2(FIGURE,BUNDLE/'memory_figure.png')
manifest={'artifact':'srq_fly_priority5_whole_process_memory','uses_test_set':False,'git_commit':subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip(),'source_hashes':EXPECTED,'result_sha256':sha(RESULT)}
(BUNDLE/'manifest.json').write_text(json.dumps(manifest,indent=2))
ARCHIVE=Path(shutil.make_archive(str(BUNDLE),'zip',root_dir=BUNDLE.parent,base_dir=BUNDLE.name))
print('ZIP:',ARCHIVE,'bytes=',ARCHIVE.stat().st_size,'sha256=',sha(ARCHIVE))
from google.colab import files
files.download(str(ARCHIVE))